In [1]:
#| default_exp lawa

In [2]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:80: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["VLLM_ALLOW_DEPRECATED_BEAM_SEARCH"] = "1"
#os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [4]:
#| export
from os import getenv
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from front.common import process_seq
import random
model_path = getenv("MODEL")

In [5]:
model_path = 'large/poetry'
model_path = 'large/pelevin'

# loss 1.43 for llama 3.2 1B


model_path = 'xl/pelevin'

model_path = 'lawa'


In [6]:
#| export
full_path = f'./models/{model_path}'
tokenizer = AutoTokenizer.from_pretrained(full_path)
model = LLM(model=full_path, dtype="float16", device="cuda", gpu_memory_utilization=0.40)

INFO 10-01 08:31:32 config.py:1652] Downcasting torch.float32 to torch.float16.
INFO 10-01 08:31:32 llm_engine.py:226] Initializing an LLM engine (v0.6.1.dev238+ge2c6e0a82) with config: model='./models/large/pelevin', speculative_config=None, tokenizer='./models/large/pelevin', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=./models/large/pelevin, use_v2_

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


INFO 10-01 08:31:33 model_runner.py:1014] Starting to load model ./models/large/pelevin...


[W1001 08:31:32.766346300 socket.cpp:697] [c10d] The client socket cannot be initialized to connect to [bbb]:47071 (errno: 97 - Address family not supported by protocol).


Loading pt checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


/usr/local/lib/python3.12/dist-packages/vllm/model_executor/model_loader/weight_utils.py:424: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(bin_file, map_

INFO 10-01 08:31:42 model_runner.py:1025] Loading model weights took 1.4419 GB
INFO 10-01 08:31:42 gpu_executor.py:122] # GPU blocks: 2637, # CPU blocks: 1456
INFO 10-01 08:31:44 model_runner.py:1329] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 10-01 08:31:44 model_runner.py:1333] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 10-01 08:31:58 model_runner.py:1456] Graph capturing finished in 15 secs.


In [ ]:
model.llm_engine.model_config.max_model_len

In [7]:
#| export
def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

stop_token_ids = iftoken(tokenizer, ['<|endoftext|>','|eot_id|','<|end_of_text|>','<|eot_id|>','<pad>'])

def create_token_blocker(tokenizer, blocked_tokens):
    blocked_token_ids = iftoken(tokenizer, blocked_tokens)
    
    def token_blocker(input_ids, scores):
        if scores.dim() == 2:
            scores[:, list(blocked_token_ids)] = -float('inf')
        elif scores.dim() == 1:
            scores[list(blocked_token_ids)] = -float('inf')
        else:
            raise ValueError(f"Unexpected score tensor shape: {scores.shape}")
        return scores
    
    return token_blocker
    
def get_sampling_params(tokenizer, length: int, num_samples: int, allow_linebreak: bool, temperature: float):
    blocked_tokens = ['.\n','\n\t\t','http://',',[','("','.]',' («',')','\u2004',']','(«','[', ' [', '(', ' (', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    
    token_blocker = create_token_blocker(tokenizer, blocked_tokens)
    return SamplingParams(
        max_tokens=length,
        n=num_samples,
        top_k=-1,
        stop_token_ids=stop_token_ids,
        ignore_eos=True,
        logits_processors=[token_blocker],
        repetition_penalty=2.,
        temperature=temperature,
        top_p=0.9,
        seed=random.randint(0, 1000000),
#        use_beam_search=True,
#        best_of=num_samples*4,
#        temperature=0.,
#        top_p=1.,
         #penalty_alpha=0.6, top_k=4
    )

In [8]:
stop_token_ids

[50257]

In [10]:
#| export
def get_sample(prompt: str, length: int, num_samples: int, allow_linebreak: bool, temperature: float = 0.2):
    max_input = model.llm_engine.model_config.max_model_len - length
    prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')
    sampling_params = get_sampling_params(tokenizer, length, num_samples, allow_linebreak, temperature)
    outputs = model.generate(prompt, sampling_params)
    generated_sequences = [oo.text for o in outputs for oo in o.outputs]
    return process_seq(generated_sequences)

In [15]:
%%time
get_sample('На словах ты Лев Толстой, а на деле'*100000, 300, 4, False)

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.75s/it, est. speed input: 414.02 toks/s, output: 686.22 toks/s]

CPU times: user 4.44 s, sys: 191 ms, total: 4.63 s
Wall time: 4.62 s


[' Ты Львенок Толстого. А потом я стал думать о том что бы такое написать и придумал такую фразу: «Я не знаю кто такой этот самый Лева Толст». И тут же мне стало ясно как он выглядит в жизни — это был просто толстый человек с большим животом» / Пелевин В., Битов Р./ Мифы русского рока/ Сост.: Ерофеев Д.; Художественный редактор-составитель Нелли Коган; Предисловие Елены Шубиной // http:// www magazines russes amp;. 2000/. • * Оригинал цитаты смонтирован из фрагментов интервью Льва Николаевича со Львом Николаевичем по телефону во время его визита к нему домой летом 1910 года.]. Но если вы хотите узнать больше об этом человеке или хотя б понять почему именно так произошло то вам надо будет прочесть книгу самого писателя.[ 5 - Смена названия книги произошла после того момента когда она была опубликована под названием« Записки сумасшедшего».] Это очень интересная книга! Она написана человеком совершенно необычным для нашего времени… Он пишет ее всю жизнь только потому чтобы рассказать людя